<a href="https://colab.research.google.com/github/gamasand/sismos-mexico/blob/main/titanic_pendiente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **título pendiente**

## 1. Introducción
...

## 2. Carga y exploración de entrada
### 2.1 Las bibliotecas
Utilizaremos Pandas para manejar la tabla de dato, Matplotlib para los histogramas y Seaborn para las barras y el mapa de color; las importaremos de un jalón.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")



### 2.2 El dataset
La fuente esta en...

In [3]:
# Indicamos dónde buscaremos el archivo si está en local:
ruta = Path("..") / "data" / "titanic.csv"
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if ruta.exists():
    df_original = pd.read_csv(ruta)
    print("Archivo utilizado:", ruta)
else:
    df_original = pd.read_csv(url)
    print("Train.csv jalado desde Kaggle")

print("Filas y columnas:", df_original.shape)
df_original.head()

Train.csv jalado desde Kaggle
Filas y columnas: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 2.3 ¿Cómo vienen los datos?
Tenemos que revisar las características, sus tipos y ver un resumen de las variables que son numéricas.

In [4]:
print("Columnas:", df_original.columns.tolist())
df_original.info()

Columnas: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Percibimos que es un dataset muy limpio. Tenemos valores nulos en la edad, por lo que deberemos explorar el porqué de esta presencia. Una primera hipótesis sería relacionar edades faltantes con dos casos en torno a la característica que nos indica si sobrevivió:

- Sobrevivieron, pero fue tanto el trauma que se desentendieron de la situación mediática y no reportaron datos como la edad.

- Fallecieron y no tenían familiares o gente cercana que pudiera corroborar la información de la persona fallecida.

En el número de cabina registrada hay muchos faltantes, destacando con una gran presencia de faltantes.

Embarked tiene 2 valores faltantes, por lo más tarde analizaremos qué hacer con esto.


In [5]:
df_original.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Asumiendo la naturaleza binaria de la columna de *Survived*, se puede percibir que un 38.38 % de las personas la libraron. Es también triste ver numéricamente la tragedia, pues más de la mitad de las personas perdieron la vida.

### 2.4 Diccionario de datos

Es una primera visión, en dado caso que necesitemos inputar faltantes, recategorizar o hacer movimientos.

| Variable | Descripción | Tipo de variable | Valores o rango | Relevancia para el análisis |
|---|---|---|---|---|
| `PassengerId` | Identificador de cada pasajero. | Categórica, identificador nominal. | Enteros únicos del 1 al 891. | Permite identificar registros y revisar duplicados; su número no representa una característica del pasajero. |
| `Survived` | Indica si el pasajero sobrevivió. | Categórica binaria. | 0 = no sobrevivió; 1 = sobrevivió. | Es la variable principal que compararemos entre grupos. |
| `Pclass` | Clase del boleto. | Categórica ordinal. | 1 = primera; 2 = segunda; 3 = tercera. | Permite comparar supervivencia entre clases del boleto. |
| `Name` | Nombre registrado del pasajero. | Categórica nominal, texto de identificación. | Nombres de los pasajeros; 891 valores distintos. | Sirve para reconocer registros; no analizaremos cada nombre como un grupo de pasajeros. |
| `Sex` | Sexo registrado del pasajero. | Categórica nominal. | `female`, `male`. | Permite comparar la proporción de supervivientes según el sexo registrado. |
| `Age` | Edad en años. | Numérica continua. | De 0.42 a 80 años, además de faltantes. | Permite explorar si la edad se asocia con la supervivencia. Las fracciones de año pueden corresponder a bebés. |
| `SibSp` | Número de hermanos y/o cónyuges a bordo. | Numérica discreta. | De 0 a 8. | Describe parte del acompañamiento familiar del pasajero. |
| `Parch` | Número de padres y/o hijos a bordo. | Numérica discreta. | De 0 a 6. | Complementa la información sobre familiares a bordo. |
| `Ticket` | Código del boleto. | Categórica nominal. | Códigos alfanuméricos; algunos boletos se repiten. | Puede identificar pasajeros que comparten boleto; no representa una cantidad. |
| `Fare` | Tarifa registrada del boleto, en libras esterlinas. | Numérica continua. | De 0 a 512.3292. | Permite explorar la asociación entre la tarifa y la supervivencia; también está relacionada con la clase del boleto. |
| `Cabin` | Cabina o cabinas registradas. | Categórica nominal. | Códigos como `C85`, `C23 C25 C27` y valores faltantes. | Aporta información de alojamiento, pero la gran cantidad de faltantes limita su análisis. |
| `Embarked` | Puerto de embarque. | Categórica nominal. | C = Cherbourg; Q = Queenstown; S = Southampton; también hay faltantes. | Permite comparar supervivencia entre puertos de embarque. |


## 3. Limpieza y preparación de los datos
### 3.1 Valores faltantes

Primero contaremos los faltantes y calcularemos qué porcentaje representan en cada columna para determinar su importancia en el análisis del dataset.

In [6]:
faltantes = pd.DataFrame({
    "Faltantes": df_original.isna().sum(),
    "Porcentaje (%)": 100 * df_original.isna().sum() / len(df_original)
})

faltantes.round(2)

,Faltantes,Porcentaje (%)
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
Age,177,19.87
SibSp,0,0.00
Parch,0,0.00
Ticket,0,0.00
Fare,0,0.00


In [16]:
sin_edad_vivo = df_original[
    df_original["Age"].isna() & (df_original["Survived"] == 1)
].shape[0]

sin_edad_muerto = df_original[
    df_original["Age"].isna() & (df_original["Survived"] == 0)
].shape[0]

print(f"Sin edad, pero sobrevivieron: {sin_edad_vivo}")
print(f"Sin edad, pero fallecieron: {sin_edad_muerto}")

Sin edad, pero sobrevivieron: 52
Sin edad, pero fallecieron: 125


Como no hay un sesgo claro en cuanto a por qué tenemos faltantes para aplicar una conjetura, lo mejor será no hacer suposiciones sobre la coliumna y aplicar un método distinto para tratarlos.

### 3.2 ¿Qué haremos con los faltantes?

| Variable | Decisión | Justificación |
|---|---|---|
| `Age` | Completar con la mediana de las edades disponibles. | Conservaremos los pasajeros sin edad registrada. La mediana es una medida sencilla y menos sensible a valores extremos que la media; no debe confundirse con la edad real de esos pasajeros. |
| `Embarked` | Completar con la moda. | Solo hay dos faltantes y utilizaremos el puerto más frecuente. Es una aproximación, por lo que esas dos asignaciones no son datos observados. |
| `Cabin` | Completar con la etiqueta `Sin registro`. | Faltan demasiadas cabinas para asignar una específica. La etiqueta identifica la ausencia de información y no una cabina real. |



### 3.3 Copia de trabajo y consistencia de los textos

Crearemos df como una copia de trabajo. Le quitaremos espacios al principio y al final de los textos y dejaremos una misma escritura para el sexo y el puerto de embarque.

In [17]:
df = df_original.copy()

for columna in ["Name", "Sex", "Ticket", "Cabin", "Embarked"]:
    df[columna] = df[columna].astype("string").str.strip().replace("", pd.NA)

df["Sex"] = df["Sex"].str.lower()
df["Embarked"] = df["Embarked"].str.upper()

print("Valores de Sex:", df["Sex"].dropna().unique())
print("Valores de Embarked:", df["Embarked"].dropna().unique())

Valores de Sex: <StringArray>
['male', 'female']
Length: 2, dtype: string
Valores de Embarked: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: string


### 3.4 Duplicados e inconsistencias

Si bien se tiene un identificador primario por pasajero, vale la pena revisar las filas para evitar duplicados. El mismo apellido, boleto y cabina no necesariamente es un duplicado.

In [18]:
print("Filas duplicadas:", df.duplicated().sum())
print("PassengerId repetidos:", df["PassengerId"].duplicated().sum())

# Si se repitiera una fila completa, conservaríamos una sola copia:
df = df.drop_duplicates().copy()

Filas duplicadas: 0
PassengerId repetidos: 0


In [22]:
# Contamos valores fuera de los rangos o categorías esperadas:
inconsistencias = pd.Series({
    "Survived fuera de 0 y 1": (~df["Survived"].isin([0, 1])).sum(),
    "Pclass fuera de 1, 2 y 3": (~df["Pclass"].isin([1, 2, 3])).sum(),
    "Sex fuera de female y male": (~df["Sex"].isin(["female", "male"])).sum(),
    "Embarked fuera de C, Q y S": (
        df["Embarked"].notna() & ~df["Embarked"].isin(["C", "Q", "S"])
    ).sum(),
    "Age fuera de 0 a 100 años": (
        df["Age"].notna() & ~df["Age"].between(0, 100)
    ).sum(),
    "SibSp negativos o no enteros": ((df["SibSp"] < 0) | (df["SibSp"] % 1 != 0)).sum(),
    "Parch negativos o no enteros": ((df["Parch"] < 0) | (df["Parch"] % 1 != 0)).sum(),
    "Fare negativas o sin costo": (df["Fare"] <= 0).sum()
}, name="Casos")

inconsistencias.to_frame()

,Casos
Survived fuera de 0 y 1,0
"Pclass fuera de 1, 2 y 3",0
Sex fuera de female y male,0
"Embarked fuera de C, Q y S",0
Age fuera de 0 a 100 años,0
SibSp negativos o no enteros,0
Parch negativos o no enteros,0
Fare negativas o sin costo,15


En este dataset no aparecen filas duplicadas ni valores fuera de las categorías y rangos revisados, a excepción de la tarifa. La limpieza de textos evita diferencias por espacios o mayúsculas. Las edades menores a un año se conservarán como fracciones, sin redondearlasa 1.

Revisaremos las tarifas iguales a cero. Son casos que llaman la atención, pero sin información adicional no podemos afirmar que sean errores ni asignarles otra tarifa, pues pudierantrarse de regalos.

In [20]:
tarifas_cero = df[df["Fare"] == 0]

print("Pasajeros con tarifa igual a cero:", len(tarifas_cero))
tarifas_cero[["PassengerId", "Name", "Pclass", "Fare"]].head(10)

Pasajeros con tarifa igual a cero: 15


,PassengerId,Name,Pclass,Fare
179,180,"Leonard, Mr. Lionel",3,0.0
263,264,"Harrison, Mr. William",1,0.0
271,272,"Tornquist, Mr. William Henry",3,0.0
277,278,"Parkes, Mr. Francis ""Frank""",2,0.0
302,303,"Johnson, Mr. William Cahoone Jr",3,0.0
413,414,"Cunningham, Mr. Alfred Fleming",2,0.0
466,467,"Campbell, Mr. William",2,0.0
481,482,"Frost, Mr. Anthony Wood ""Archie""",2,0.0
597,598,"Johnson, Mr. Alfred",3,0.0
633,634,"Parr, Mr. William Henry Marsh",1,0.0


### 3.5 ¿Y los valores atípicos?

Usaremos la regla del **rango intercuartílico (IQR)**. Para cada variable calcularemos el primer y tercer cuartil; marcaremos los valores menores a `Q1 - 1.5 * IQR` o mayores a `Q3 + 1.5 * IQR`.


In [24]:
numericas = ["Age", "SibSp", "Parch", "Fare"]
revision_atipicos = []

for columna in numericas:
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    casos = (df[columna] < limite_inferior) | (df[columna] > limite_superior)
    revision_atipicos.append({
        "Variable": columna,
        "Límite inferior": limite_inferior,
        "Límite superior": limite_superior,
        "Casos señalados": casos.sum()
    })

pd.DataFrame(revision_atipicos).round(2)

,Variable,Límite inferior,Límite superior,Casos señalados
0,Age,-6.69,64.81,11
1,SibSp,-1.50,2.50,46
2,Parch,0.00,0.00,213
3,Fare,-26.72,65.63,116
